In [1]:
from datasets import load_dataset

# Пример загрузки
dataset = load_dataset("cardiffnlp/tweet_eval", "sentiment")

# Сохраняем каждый сплит отдельно
for split in dataset.keys():
    dataset[split].to_csv(f"tweet_eval_{split}.csv", index=False)


Creating CSV from Arrow format:   0%|          | 0/46 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/13 [00:00<?, ?ba/s]

Creating CSV from Arrow format:   0%|          | 0/2 [00:00<?, ?ba/s]

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# Загружаем LLM (можно маленькую модель типа MPT-7B или LLaMA)
model_name = "meta-llama/Llama-2-7b-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=256)

dataset = [
    {"text": "Мне понравился этот фильм", "label": "Положительный"},
    {"text": "Фильм скучный и затянутый", "label": "Отрицательный"}
]

for entry in dataset:
    prompt = f"""
        Текст: "{entry['text']}"
        Метка: "{entry['label']}"
        Задача: Объясни, почему текст относится к этой категории, кратко.
        """
        
    output = generator(prompt, max_new_tokens=64, do_sample=True, temperature=0.7)[0]["generated_text"]
    
    # вырезаем только объяснение (можно через split по метке)
    explanation = output.split("Задача:")[-1].strip()
    entry["explanation"] = explanation

print(dataset)


In [ ]:
import pandas as pd 

df = pd.read_csv("tweet_eval_train.csv")

In [3]:
df.head()

,text,label
0,@user @user what do these '1/2 naked pics' hav...,1
1,OH: “I had a blue penis while I was this” [pla...,1
2,"@user @user That's coming, but I think the vic...",1
3,I think I may be finally in with the in crowd ...,2
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",0


In [4]:
count = 0

for index, row in df.iterrows():
    text = row['text']
    label = row['label']
    print(index, text, label)
    count += 1
    if count == 5:
        break

0 @user @user what do these '1/2 naked pics' have to do with anything? They're not even like that. 1
1 OH: “I had a blue penis while I was this” [playing with Google Earth VR] 1
2 @user @user That's coming, but I think the victims are going to be Medicaid recipients. 1
3 I think I may be finally in with the in crowd #mannequinchallenge  #grads2014 @user 2
4 @user Wow,first Hugo Chavez and now Fidel Castro. Danny Glover, Michael Moore, Oliver Stone, and Sean Penn are running out of heroes. 0


In [6]:
import pandas as pd
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

model_name = "NousResearch/Nous-Hermes-7b"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

dataset = pd.read_csv("tweet_eval_validation.csv")

# Создаём новую колонку для объяснений
dataset["explanation"] = ""

generator = pipeline("text-generation", model=model, tokenizer=tokenizer, max_length=256)

# Проходим по строкам датасета
for row in dataset.itertuples(index=True):
    text = row.text
    label = row.label

    prompt = f"""
        Text: "{text}"
        Label: "{label}"
        Objective: Briefly explain why the text belongs to this category.
        """
    

    output = generator(prompt, max_new_tokens=64, do_sample=True, temperature=0.7)[0]["generated_text"]
    
    explanation = output.split("Задача:")[-1].strip()
    
    dataset.at[row.Index, "explanation"] = explanation

dataset.to_csv("tweet_eval_train_with_explanations.csv", index=False)

print("Готово! Объяснения сгенерированы и сохранены.")


OSError: NousResearch/Nous-Hermes-7b is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `hf auth login` or by passing `token=<your_token>`

In [1]:
from src.data.dataset import load_tweet_dataset
from src.training.configs import TrainingConfig, get_lora_config

In [2]:
config =  get_lora_config()

In [5]:
config

TrainingConfig(batch_size=16, learning_rate=0.0005, num_epochs=3, max_length=256, weight_decay=0.01, max_grad_norm=1.0, gradient_accumulation_steps=1, lora_rank=8, lora_alpha=16, target_modules=('k_proj', 'v_proj'), eval_batch_size=100, eval_steps=50, device='auto')

In [ ]:
from src.data.dataset import load_tweet_dataset
from src.data.preprocessing import DataProcessor, TokenizerWrapper, preprocess_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

In [2]:
dataset = load_tweet_dataset(cache_dir="~/.cache/huggingface/datasets")

MODEL_NAME = "OuteAI/Lite-Oute-1-300M-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

In [21]:
processed_dataset = {}

In [3]:
for split, data in dataset.items():
    proc_dataset[split] = data.map(process_example, fn_kwargs={"tokenizer": tokenizer})

NameError: name 'process_example' is not defined

In [4]:
proc_dataset = preprocess_dataset(dataset, tokenizer)

Tokenizing train:   0%|          | 0/45615 [00:00<?, ? examples/s]

Formatting test:   0%|          | 0/12284 [00:00<?, ? examples/s]

Tokenizing test:   0%|          | 0/12284 [00:00<?, ? examples/s]

Formatting validation:   0%|          | 0/2000 [00:00<?, ? examples/s]

Tokenizing validation:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [9]:
proc_dataset

{'train': Dataset({
     features: ['text', 'label', 'str_label', 'prompt', 'full_prompt', 'input_ids', 'attention_mask', 'full_input_ids', 'full_attention_mask'],
     num_rows: 45615
 }),
 'test': Dataset({
     features: ['text', 'label', 'str_label', 'prompt', 'full_prompt', 'input_ids', 'attention_mask', 'full_input_ids', 'full_attention_mask'],
     num_rows: 12284
 }),
 'validation': Dataset({
     features: ['text', 'label', 'str_label', 'prompt', 'full_prompt', 'input_ids', 'attention_mask', 'full_input_ids', 'full_attention_mask'],
     num_rows: 2000
 })}

In [6]:
proc_dataset["train"]["text"][1]

'"Ben Smith / Smith (concussion) remains out of the lineup Thursday, Curtis #NHL #SJ"'

In [12]:
len(proc_dataset["train"]["full_input_ids"][1])

138

In [ ]:
from functools import partial
from torch.utils.data import DataLoader

In [ ]:
dataloader = DataLoader(
    dataset["train"],
    batch_size=2,
    shuffle=True,
    collate_fn=partial(pad_collate_fn, pad_token_id=tokenizer.pad_token_id),
)
next(iter(dataloader))

In [ ]:
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

# Инициализация
tokenizer_wrapper = TokenizerWrapper(model_name)

# Предположим, у нас есть сырой датасет
raw_dataset = {
    "train": [...],  # ваш тренировочный датасет
    "validation": [...]  # ваш валидационный датасет
}

# Предобработка датасета
processed_dataset = preprocess_dataset(
    dataset=raw_dataset,
    tokenizer=tokenizer_wrapper.tokenizer,
    system_prompt="You are a helpful assistant.",
    max_length=256
)

# Создаем collate функцию
collate_fn = tokenizer_wrapper.get_collate_fn(
    pad_token_id=tokenizer_wrapper.tokenizer.pad_token_id
)

# Создаем DataLoader'ы
train_dataloader = DataLoader(
    processed_dataset["train"],
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
)

# Проверяем работу
for batch in train_dataloader:
    print("Batch shapes:")
    print(f"  input_ids: {batch['input_ids'].shape}")
    print(f"  attention_mask: {batch['attention_mask'].shape}")
    print(f"  labels: {batch['labels'].shape}")
    
    # Проверяем паддинг
    print(f"  Real sequence lengths: {(batch['attention_mask'] == 1).sum(dim=1)}")
    break  # только первый батч

In [2]:
from src.data.dataset import load_tweet_dataset
from src.data.preprocessing import TokenizerWrapper, preprocess_dataset
from src.models.base import setup_device, apply_peft_to_model, freeze_layers
from transformers import AutoModelForCausalLM
from torch.utils.data import DataLoader

MODEL_NAME = "OuteAI/Lite-Oute-1-300M-Instruct"

model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
tokenizer_wrapper = TokenizerWrapper(MODEL_NAME)

device = setup_device()
model.to(device)

dataset = load_tweet_dataset(cache_dir="~/.cache/huggingface/datasets")

processed_dataset = preprocess_dataset(dataset, tokenizer_wrapper.tokenizer)

collate_fn = tokenizer_wrapper.get_collate_fn(
    pad_token_id=tokenizer_wrapper.tokenizer.pad_token_id
)

train_dataloader = DataLoader(
    processed_dataset["train"],
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn,
)


In [2]:
for batch in train_dataloader:
    print("Batch shapes:")
    print(f"  input_ids: {batch['input_ids'].shape}")
    print(f"  attention_mask: {batch['attention_mask'].shape}")
    print(f"  labels: {batch['labels'].shape}")
    
    # Проверяем паддинг
    print(f"  Real sequence lengths: {(batch['attention_mask'] == 1).sum(dim=1)}")
    break  # только первый батч

Batch shapes:
  input_ids: torch.Size([8, 164])
  attention_mask: torch.Size([8, 164])
  labels: torch.Size([8, 164])
  Real sequence lengths: tensor([143, 126, 158, 130, 128, 141, 153, 164])


In [16]:
train_dataloader

In [3]:
trainable_params = [
    p for n, p in model.named_parameters() 
    if p.requires_grad
]

In [8]:
trainable_params

[Parameter containing:
 tensor([[ 0.0000,  0.0000,  0.0000,  ...,  0.0000,  0.0000,  0.0000],
         [ 0.2051,  0.0825, -0.0947,  ..., -0.0059, -0.1406,  0.0041],
         [-0.0854,  0.0042, -0.0281,  ..., -0.1494, -0.1226, -0.0884],
         ...,
         [ 0.0518, -0.0022,  0.0016,  ...,  0.0420, -0.0041,  0.0007],
         [-0.0388,  0.0153, -0.0378,  ...,  0.0170, -0.0036,  0.0082],
         [-0.0225,  0.0148, -0.0226,  ...,  0.0099, -0.0111,  0.0045]],
        device='mps:0', requires_grad=True),
 Parameter containing:
 tensor([[-0.0071,  0.0195,  0.0614,  ...,  0.0493,  0.0882, -0.0642],
         [ 0.0784,  0.0924, -0.0176,  ..., -0.0304, -0.0625,  0.2094],
         [ 0.0160,  0.0803, -0.0071,  ...,  0.0780, -0.0452, -0.0141],
         ...,
         [-0.1493,  0.0192,  0.1065,  ..., -0.0047, -0.0202,  0.0202],
         [ 0.0631, -0.1480, -0.0523,  ...,  0.0605, -0.0062, -0.0561],
         [-0.0084,  0.0030, -0.0504,  ...,  0.0955, -0.1592,  0.1057]],
        device='mps:0', req

In [14]:
import torch 

In [15]:
def get_optimizer(model, learning_rate: float, weight_decay: float = 0.01):
    """
    Create optimizer for PEFT training.
    
    Args:
        model: Model with parameters
        learning_rate: Learning rate
        weight_decay: Weight decay for optimizer
        
    Returns:
        Configured optimizer
    """
    trainable_params = [
        p for n, p in model.named_parameters() 
        if p.requires_grad
    ]
    
    return torch.optim.AdamW(
        trainable_params,
        lr=learning_rate,
        weight_decay=weight_decay,
        betas=(0.9, 0.999),
        eps=1e-8
    )

In [16]:
get_optimizer(model, learning_rate=0.01)

AdamW (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: True
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.01
    maximize: False
    weight_decay: 0.01
)